In [4]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load VIIRS data for each year
viirs_2021 = pd.read_csv('Maldives_light_intensity_2021.csv')
viirs_2022 = pd.read_csv('Maldives_light_intensity_2022.csv')
viirs_2023 = pd.read_csv('Maldives_light_intensity_2023.csv')

# Add a 'year' column to each dataset
viirs_2021['year'] = 2021
viirs_2022['year'] = 2022
viirs_2023['year'] = 2023


In [5]:
# Combine the datasets
viirs_data = pd.concat([viirs_2021, viirs_2022, viirs_2023], ignore_index=True)


In [6]:
# Define approximate latitude and longitude bounds for Maldives
min_lat, max_lat = -1.0, 8.0
min_lon, max_lon = 72.5, 74.0

# Filter the data
maldives_viirs = viirs_data[
    (viirs_data['latitude'] >= min_lat) &
    (viirs_data['latitude'] <= max_lat) &
    (viirs_data['longitude'] >= min_lon) &
    (viirs_data['longitude'] <= max_lon)
]


In [7]:
# Create geometry column (points based on latitude and longitude)
geometry = [Point(xy) for xy in zip(maldives_viirs['longitude'], maldives_viirs['latitude'])]

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(maldives_viirs, geometry=geometry, crs="EPSG:4326")  # EPSG:4326 is the standard WGS84 CRS

# Check GeoDataFrame
print(gdf.head())


   longitude  latitude  light_intensity  year            geometry
0  72.637500       0.0              0.0  2021   POINT (72.6375 0)
1  72.641667       0.0              0.0  2021  POINT (72.64167 0)
2  72.645833       0.0              0.0  2021  POINT (72.64583 0)
3  72.650000       0.0              0.0  2021     POINT (72.65 0)
4  72.654167       0.0              0.0  2021  POINT (72.65417 0)


In [11]:
# Step 1: Group by year to calculate the average radiance
annual_radiance = gdf.groupby('year')['light_intensity'].mean().reset_index()

# Step 2: Calculate percentage change
annual_radiance['percentage_change'] = annual_radiance['light_intensity'].pct_change() * 100

# Step 3: Merge the annual radiance and percentage change back into the original VIIRS dataset
gdf = gdf.merge(annual_radiance, on='year', how='left')

# Step 4: Save the combined file
gdf.to_file('maldives_viirs_combined.shp')  # Save as SHP file
gdf.to_csv('maldives_viirs_combined.csv', index=False)  # Save as CSV file

print("Combined VIIRS dataset saved as both SHP and CSV.")



/var/folders/3y/2hr2rcnx7xl_2_w54n12rq6c0000gn/T/ipykernel_88083/755625592.py:11: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file('maldives_viirs_combined.shp')  # Save as SHP file
/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'light_intensity_x' to 'light_inte'
  ogr_write(
/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'light_intensity_y' to 'light_in_1'
  ogr_write(
/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'percentage_change' to 'percentage'
  ogr_write(


Combined VIIRS dataset saved as both SHP and CSV.
